In [1]:
# =========================================
# IMPORT LIBRARIES
# =========================================

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

import boto3
from botocore.client import Config

In [2]:
# =========================================
# INIT SPARK SESSION
# =========================================

spark = SparkSession.builder \
    .appName("SV3_Altcoins_ETL") \
    .getOrCreate()

print("Spark Started Successfully")

# =========================================
# MINIO S3A CONFIG
# =========================================

hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()

hadoop_conf.set("fs.s3a.endpoint", "http://minio:9000")
hadoop_conf.set("fs.s3a.access.key", "admin")
hadoop_conf.set("fs.s3a.secret.key", "password123")
hadoop_conf.set("fs.s3a.path.style.access", "true")
hadoop_conf.set("fs.s3a.connection.ssl.enabled", "false")

hadoop_conf.set(
    "fs.s3a.aws.credentials.provider",
    "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider"
)

print("MinIO Configuration Completed")

Spark Started Successfully
MinIO Configuration Completed


In [3]:
# =========================================
# LOAD RAW DATA FROM MINIO
# =========================================

df = spark.read.csv(
    "s3a://crypto-raw-data/altcoins_500d.csv",
    header=True,
    inferSchema=True
)

print("Raw Dataset Loaded")

print("Total Rows:", df.count())

df.show(20, False)

df.printSchema()

Raw Dataset Loaded
Total Rows: 7000
+-------+----------+------+------+------+------+--------------+
|symbol |timestamp |open  |high  |low   |close |volume        |
+-------+----------+------+------+------+------+--------------+
|ETH/USD|2025-01-23|3238.4|3346.6|3183.0|3339.1|6448.56804778 |
|ETH/USD|2025-01-24|3341.4|3427.4|3275.9|3311.0|6021.69432698 |
|ETH/USD|2025-01-25|3309.7|3350.0|3269.7|3317.9|2165.65298635 |
|ETH/USD|2025-01-26|3320.9|3359.6|3229.5|3231.3|3371.92050419 |
|ETH/USD|2025-01-27|3228.0|3251.5|3024.0|3181.9|16821.74301423|
|ETH/USD|2025-01-28|3181.7|3224.9|3039.0|3075.8|4970.19146861 |
|ETH/USD|2025-01-29|3077.6|3181.1|3055.0|3114.2|4377.18334445 |
|ETH/USD|2025-01-30|3114.3|3283.1|3092.2|3247.8|5651.86284548 |
|ETH/USD|2025-01-31|3247.2|3437.9|3214.0|3300.1|8634.67643534 |
|ETH/USD|2025-02-01|3299.5|3331.5|3101.6|3116.8|3380.51704713 |
|ETH/USD|2025-02-02|3115.0|3162.5|2751.0|2869.1|14960.17996076|
|ETH/USD|2025-02-03|2869.6|2923.0|2118.0|2883.2|46609.82125989|
|ETH

In [4]:
# =========================================
# NULL CHECK
# =========================================

print("NULL CHECK")

df.select([
    F.count(
        F.when(F.col(c).isNull(), c)
    ).alias(c)
    for c in df.columns
]).show()

# =========================================
# DUPLICATE CHECK
# =========================================

total_rows = df.count()

unique_rows = df.dropDuplicates(
    ["symbol", "timestamp"]
).count()

print("Total Rows:", total_rows)
print("Unique Rows:", unique_rows)
print("Duplicates:", total_rows - unique_rows)

NULL CHECK
+------+---------+----+----+---+-----+------+
|symbol|timestamp|open|high|low|close|volume|
+------+---------+----+----+---+-----+------+
|     0|        0|   0|   0|  0|    0|     0|
+------+---------+----+----+---+-----+------+

Total Rows: 7000
Unique Rows: 7000
Duplicates: 0


In [5]:
# =========================================
# STANDARDIZE TIMESTAMP
# =========================================

df = df.withColumn(
    "timestamp",
    F.to_timestamp("timestamp")
)

df = df.orderBy(
    "symbol",
    "timestamp"
)

df.select(
    F.min("timestamp").alias("min_time"),
    F.max("timestamp").alias("max_time")
).show(truncate=False)

+-------------------+-------------------+
|min_time           |max_time           |
+-------------------+-------------------+
|2025-01-23 00:00:00|2026-06-06 00:00:00|
+-------------------+-------------------+



In [6]:
# =========================================
# GAP DETECTION
# =========================================

w = Window.partitionBy(
    "symbol"
).orderBy(
    "timestamp"
)

df = df.withColumn(
    "prev_time",
    F.lag("timestamp").over(w)
)

df = df.withColumn(
    "diff_day",
    F.datediff(
        F.col("timestamp"),
        F.col("prev_time")
    )
)

df.select(
    "symbol",
    "timestamp",
    "prev_time",
    "diff_day"
).show(20, False)

gap_count = df.filter(
    F.col("diff_day") > 1
).count()

print("Gap Count:", gap_count)

+--------+-------------------+-------------------+--------+
|symbol  |timestamp          |prev_time          |diff_day|
+--------+-------------------+-------------------+--------+
|AAVE/USD|2025-01-23 00:00:00|NULL               |NULL    |
|AAVE/USD|2025-01-24 00:00:00|2025-01-23 00:00:00|1       |
|AAVE/USD|2025-01-25 00:00:00|2025-01-24 00:00:00|1       |
|AAVE/USD|2025-01-26 00:00:00|2025-01-25 00:00:00|1       |
|AAVE/USD|2025-01-27 00:00:00|2025-01-26 00:00:00|1       |
|AAVE/USD|2025-01-28 00:00:00|2025-01-27 00:00:00|1       |
|AAVE/USD|2025-01-29 00:00:00|2025-01-28 00:00:00|1       |
|AAVE/USD|2025-01-30 00:00:00|2025-01-29 00:00:00|1       |
|AAVE/USD|2025-01-31 00:00:00|2025-01-30 00:00:00|1       |
|AAVE/USD|2025-02-01 00:00:00|2025-01-31 00:00:00|1       |
|AAVE/USD|2025-02-02 00:00:00|2025-02-01 00:00:00|1       |
|AAVE/USD|2025-02-03 00:00:00|2025-02-02 00:00:00|1       |
|AAVE/USD|2025-02-04 00:00:00|2025-02-03 00:00:00|1       |
|AAVE/USD|2025-02-05 00:00:00|2025-02-04

In [7]:
# =========================================
# MA10 & MA60
# =========================================

w10 = Window.partitionBy(
    "symbol"
).orderBy(
    "timestamp"
).rowsBetween(-9, 0)

w60 = Window.partitionBy(
    "symbol"
).orderBy(
    "timestamp"
).rowsBetween(-59, 0)

df = df.withColumn(
    "MA10",
    F.avg("close").over(w10)
)

df = df.withColumn(
    "MA60",
    F.avg("close").over(w60)
)

df.select(
    "symbol",
    "timestamp",
    "close",
    "MA10",
    "MA60"
).show(20, False)

+--------+-------------------+------+------------------+------------------+
|symbol  |timestamp          |close |MA10              |MA60              |
+--------+-------------------+------+------------------+------------------+
|AAVE/USD|2025-01-23 00:00:00|337.67|337.67            |337.67            |
|AAVE/USD|2025-01-24 00:00:00|334.96|336.315           |336.315           |
|AAVE/USD|2025-01-25 00:00:00|325.82|332.81666666666666|332.81666666666666|
|AAVE/USD|2025-01-26 00:00:00|316.21|328.665           |328.665           |
|AAVE/USD|2025-01-27 00:00:00|301.8 |323.29200000000003|323.29200000000003|
|AAVE/USD|2025-01-28 00:00:00|281.91|316.39500000000004|316.39500000000004|
|AAVE/USD|2025-01-29 00:00:00|291.21|312.79714285714283|312.79714285714283|
|AAVE/USD|2025-01-30 00:00:00|315.76|313.1675          |313.1675          |
|AAVE/USD|2025-01-31 00:00:00|332.55|315.3211111111111 |315.3211111111111 |
|AAVE/USD|2025-02-01 00:00:00|298.25|313.61400000000003|313.61400000000003|
|AAVE/USD|20

In [8]:
# =========================================
# ROC + MOMENTUM
# =========================================

w = Window.partitionBy(
    "symbol"
).orderBy(
    "timestamp"
)

df = df.withColumn(
    "close_lag10",
    F.lag("close", 10).over(w)
)

df = df.withColumn(
    "ROC",
    (F.col("close") - F.col("close_lag10"))
    / F.col("close_lag10") * 100
)

df = df.withColumn(
    "MOM",
    F.col("close") - F.col("close_lag10")
)

df.select(
    "symbol",
    "close",
    "close_lag10",
    "ROC",
    "MOM"
).show(20, False)

+--------+------+-----------+-------------------+-------------------+
|symbol  |close |close_lag10|ROC                |MOM                |
+--------+------+-----------+-------------------+-------------------+
|AAVE/USD|337.67|NULL       |NULL               |NULL               |
|AAVE/USD|334.96|NULL       |NULL               |NULL               |
|AAVE/USD|325.82|NULL       |NULL               |NULL               |
|AAVE/USD|316.21|NULL       |NULL               |NULL               |
|AAVE/USD|301.8 |NULL       |NULL               |NULL               |
|AAVE/USD|281.91|NULL       |NULL               |NULL               |
|AAVE/USD|291.21|NULL       |NULL               |NULL               |
|AAVE/USD|315.76|NULL       |NULL               |NULL               |
|AAVE/USD|332.55|NULL       |NULL               |NULL               |
|AAVE/USD|298.25|NULL       |NULL               |NULL               |
|AAVE/USD|258.55|337.67     |-23.43116060058637 |-79.12             |
|AAVE/USD|276.35|334

In [9]:
# =========================================
# RSI 14
# =========================================

w1 = Window.partitionBy(
    "symbol"
).orderBy(
    "timestamp"
)

w14 = Window.partitionBy(
    "symbol"
).orderBy(
    "timestamp"
).rowsBetween(-13, 0)

df = df.withColumn(
    "change",
    F.col("close") - F.lag("close").over(w1)
)

df = df.withColumn(
    "gain",
    F.when(
        F.col("change") > 0,
        F.col("change")
    ).otherwise(0)
)

df = df.withColumn(
    "loss",
    F.when(
        F.col("change") < 0,
        -F.col("change")
    ).otherwise(0)
)

df = df.withColumn(
    "avg_gain",
    F.avg("gain").over(w14)
)

df = df.withColumn(
    "avg_loss",
    F.avg("loss").over(w14)
)

df = df.withColumn(
    "RS",
    F.when(
        F.col("avg_loss") == 0,
        None
    ).otherwise(
        F.col("avg_gain") /
        F.col("avg_loss")
    )
)

df = df.withColumn(
    "RSI",
    F.when(
        F.col("avg_loss") == 0,
        100
    )
    .when(
        F.col("avg_gain") == 0,
        0
    )
    .otherwise(
        100 - (100 / (1 + F.col("RS")))
    )
)

print("RSI Created")

df.select(
    "symbol",
    "timestamp",
    "close",
    "RSI"
).show(20, False)

RSI Created
+--------+-------------------+------+------------------+
|symbol  |timestamp          |close |RSI               |
+--------+-------------------+------+------------------+
|AAVE/USD|2025-01-23 00:00:00|337.67|100.0             |
|AAVE/USD|2025-01-24 00:00:00|334.96|0.0               |
|AAVE/USD|2025-01-25 00:00:00|325.82|0.0               |
|AAVE/USD|2025-01-26 00:00:00|316.21|0.0               |
|AAVE/USD|2025-01-27 00:00:00|301.8 |0.0               |
|AAVE/USD|2025-01-28 00:00:00|281.91|0.0               |
|AAVE/USD|2025-01-29 00:00:00|291.21|14.294497387027306|
|AAVE/USD|2025-01-30 00:00:00|315.76|37.77480191942861 |
|AAVE/USD|2025-01-31 00:00:00|332.55|47.59398496240602 |
|AAVE/USD|2025-02-01 00:00:00|298.25|35.99147121535181 |
|AAVE/USD|2025-02-02 00:00:00|258.55|28.070953436807102|
|AAVE/USD|2025-02-03 00:00:00|276.35|34.530776992936424|
|AAVE/USD|2025-02-04 00:00:00|272.14|33.81255866804999 |
|AAVE/USD|2025-02-05 00:00:00|259.23|31.78524986067248 |
|AAVE/USD|2025-02-0

In [10]:
# =========================================
# STOCHASTIC OSCILLATOR
# =========================================

df = df.withColumn(
    "highest_high",
    F.max("high").over(w14)
)

df = df.withColumn(
    "lowest_low",
    F.min("low").over(w14)
)

df = df.withColumn(
    "stoch_k",
    F.when(
        (
            F.col("highest_high")
            - F.col("lowest_low")
        ) == 0,
        None
    ).otherwise(
        (
            F.col("close")
            - F.col("lowest_low")
        )
        /
        (
            F.col("highest_high")
            - F.col("lowest_low")
        )
        * 100
    )
)

w3 = Window.partitionBy(
    "symbol"
).orderBy(
    "timestamp"
).rowsBetween(-2, 0)

df = df.withColumn(
    "stoch_d",
    F.avg("stoch_k").over(w3)
)

print("Stochastic Created")

df.select(
    "symbol",
    "timestamp",
    "stoch_k",
    "stoch_d"
).show(20, False)

Stochastic Created
+--------+-------------------+------------------+------------------+
|symbol  |timestamp          |stoch_k           |stoch_d           |
+--------+-------------------+------------------+------------------+
|AAVE/USD|2025-01-23 00:00:00|51.60243407707915 |51.60243407707915 |
|AAVE/USD|2025-01-24 00:00:00|30.416286842904842|41.009360459991996|
|AAVE/USD|2025-01-25 00:00:00|2.6435733819507865|28.220764767311593|
|AAVE/USD|2025-01-26 00:00:00|0.4541108986615621|11.171323707839063|
|AAVE/USD|2025-01-27 00:00:00|19.81118581032759 |7.6362900303133125|
|AAVE/USD|2025-01-28 00:00:00|1.1968258098087887|7.154040839599314 |
|AAVE/USD|2025-01-29 00:00:00|13.295173669832144|11.434395096656175|
|AAVE/USD|2025-01-30 00:00:00|45.23221022505526 |19.908069901565398|
|AAVE/USD|2025-01-31 00:00:00|67.0742812540653  |41.867221716317566|
|AAVE/USD|2025-02-01 00:00:00|22.45349291010796 |44.919994796409505|
|AAVE/USD|2025-02-02 00:00:00|10.53153153153153 |33.35310189856826 |
|AAVE/USD|2025-

In [11]:
# =========================================
# BUY / SELL LABEL
# =========================================

df = df.withColumn(
    "label",
    F.when(
        F.col("MA10") > F.col("MA60"),
        1
    ).otherwise(0)
)

print("Buy/Sell Label Created")

print("Label Distribution")

df.groupBy(
    "symbol",
    "label"
).count().show()

Buy/Sell Label Created
Label Distribution
+--------+-----+-----+
|  symbol|label|count|
+--------+-----+-----+
|AAVE/USD|    0|  366|
|AAVE/USD|    1|  134|
| ADA/USD|    0|  370|
| ADA/USD|    1|  130|
|ALGO/USD|    0|  342|
|ALGO/USD|    1|  158|
|AVAX/USD|    0|  318|
|AVAX/USD|    1|  182|
| BCH/USD|    0|  269|
| BCH/USD|    1|  231|
|DOGE/USD|    0|  327|
|DOGE/USD|    1|  173|
| DOT/USD|    0|  370|
| DOT/USD|    1|  130|
| ETH/USD|    0|  290|
| ETH/USD|    1|  210|
|LINK/USD|    0|  324|
|LINK/USD|    1|  176|
| LTC/USD|    0|  347|
| LTC/USD|    1|  153|
+--------+-----+-----+
only showing top 20 rows


In [12]:
# =========================================
# FEATURE TABLE
# =========================================

final_df = df.select(
    "symbol",
    "timestamp",
    "open",
    "high",
    "low",
    "close",
    "volume",
    "MA10",
    "MA60",
    "ROC",
    "MOM",
    "RSI",
    "stoch_k",
    "stoch_d",
    "label"
)

print(
    "Rows Before DropNA:",
    final_df.count()
)

final_df = final_df.dropna()

print(
    "Rows After DropNA:",
    final_df.count()
)

final_df.show(
    20,
    False
)

Rows Before DropNA: 7000
Rows After DropNA: 6860
+--------+-------------------+------+------+------+------+--------------+------------------+------------------+-------------------+-------------------+------------------+------------------+------------------+-----+
|symbol  |timestamp          |open  |high  |low   |close |volume        |MA10              |MA60              |ROC                |MOM                |RSI               |stoch_k           |stoch_d           |label|
+--------+-------------------+------+------+------+------+--------------+------------------+------------------+-------------------+-------------------+------------------+------------------+------------------+-----+
|AAVE/USD|2025-02-02 00:00:00|295.51|304.5 |246.86|258.55|3083.18631177 |305.70200000000006|308.6081818181819 |-23.43116060058637 |-79.12             |28.070953436807102|10.53153153153153 |33.35310189856826 |0    |
|AAVE/USD|2025-02-03 00:00:00|258.09|285.0 |192.83|276.35|3601.21212587 |299.841           

In [18]:
# # =========================================
# # DELETE OLD BUCKET
# # =========================================

# import boto3
# from botocore.client import Config

# s3 = boto3.client(
#     "s3",
#     endpoint_url="http://minio:9000",
#     aws_access_key_id="admin",
#     aws_secret_access_key="password123",
#     config=Config(signature_version="s3v4")
# )

# bucket_name = "crypto-feature-table"

# # Xóa toàn bộ object trong bucket
# objects = s3.list_objects_v2(Bucket=bucket_name)

# if "Contents" in objects:
#     for obj in objects["Contents"]:
#         s3.delete_object(
#             Bucket=bucket_name,
#             Key=obj["Key"]
#         )

# # Xóa bucket
# s3.delete_bucket(Bucket=bucket_name)

# print("Bucket Deleted")

Bucket Deleted


In [13]:
# =========================================
# CREATE MINIO BUCKET
# =========================================

s3 = boto3.client(
    "s3",
    endpoint_url="http://minio:9000",
    aws_access_key_id="admin",
    aws_secret_access_key="password123",
    config=Config(
        signature_version="s3v4"
    )
)

bucket_name = "crypto-feature-table"

if bucket_name not in [
    b["Name"]
    for b in s3.list_buckets()["Buckets"]
]:
    s3.create_bucket(
        Bucket=bucket_name
    )
    print("Bucket Created")
else:
    print("Bucket Already Exists")

Bucket Already Exists


In [14]:
# =========================================
# SAVE FEATURE TABLE
# =========================================

final_df.write \
    .mode("overwrite") \
    .parquet(
        "s3a://crypto-feature-table/altcoin_features/"
    )

print("Feature Table Saved")

Feature Table Saved


In [15]:
# =========================================
# VERIFY OUTPUT
# =========================================

verify_df = spark.read.parquet(
    "s3a://crypto-feature-table/altcoin_features/"
)

print(
    "Rows Written:",
    verify_df.count()
)

verify_df.show(
    30,
    False
)

verify_df.printSchema()

Rows Written: 6860
+--------+-------------------+------+------+------+------+--------------+------------------+------------------+-------------------+-------------------+------------------+------------------+------------------+-----+
|symbol  |timestamp          |open  |high  |low   |close |volume        |MA10              |MA60              |ROC                |MOM                |RSI               |stoch_k           |stoch_d           |label|
+--------+-------------------+------+------+------+------+--------------+------------------+------------------+-------------------+-------------------+------------------+------------------+------------------+-----+
|AAVE/USD|2025-02-02 00:00:00|295.51|304.5 |246.86|258.55|3083.18631177 |305.70200000000006|308.6081818181819 |-23.43116060058637 |-79.12             |28.070953436807102|10.53153153153153 |33.35310189856826 |0    |
|AAVE/USD|2025-02-03 00:00:00|258.09|285.0 |192.83|276.35|3601.21212587 |299.841           |305.92            |-17.497611

In [16]:
!jupyter nbconvert --to script DA_altcoins.ipynb

!python DA_altcoins.py

[NbConvertApp] Converting notebook DA_altcoins.ipynb to script
[NbConvertApp] Writing 9487 bytes to DA_altcoins.py
:: loading settings :: url = jar:file:/usr/local/spark-4.1.1-bin-hadoop3/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/jovyan/.ivy2.5.2/cache
The jars for the packages stored in: /home/jovyan/.ivy2.5.2/jars
org.postgresql#postgresql added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-2e1a8933-bfab-484c-9c9d-03f96f180cdf;1.0
	confs: [default]
	found org.postgresql#postgresql;42.6.0 in central
	found org.checkerframework#checker-qual;3.31.0 in central
	found org.apache.hadoop#hadoop-aws;3.4.2 in central
	found software.amazon.awssdk#bundle;2.29.52 in central
	found software.amazon.s3.analyticsaccelerator#analyticsaccelerator-s3;1.2.1 in central
	found org.wildfly.openssl#wildfly-openssl;2.1.4.Final in central
:: resolution report :: resolv

In [1]:
# import boto3

# s3 = boto3.client(
#     "s3",
#     endpoint_url="http://minio:9000",
#     aws_access_key_id="admin",
#     aws_secret_access_key="password123"
# )

# response = s3.list_objects_v2(
#     Bucket="crypto-raw-data"
# )

# for obj in response["Contents"]:
#     print(obj["Key"], obj["LastModified"])

altcoins_500d.csv 2026-06-06 14:58:03.066000+00:00
bitcoin_1m.csv 2026-06-10 04:38:06.392000+00:00
